# Projeto LlamaIndex - Chatbot com RAG

## Objetivos do projeto

*   Criar o processo de RAG(Retrieval Augmented Generation)
*   Dividir arquivos PDFs
*   Criar Embeddings
*   Armazenar dados em banco vetorial
*   Criação de um chatbot
*   Usar Gradio como interface
*   Langflow para fluxo de trabalho



#Pacotes e instalações

In [1]:
# LlamaIndex: 0.12.17
# Gradio: 5.16.0

In [2]:
!pip install -q llama_index
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-readers-file
!pip install -q llama-index-vector-stores-chroma
!pip install -q llama-index-llms-groq
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/7

In [3]:
from llama_index.core import SimpleDirectoryReader, StorageContext, VectorStoreIndex, load_index_from_storage
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.groq import Groq
from llama_index.core.memory import ChatSummaryMemoryBuffer
import chromadb
import pandas as pd
from google.colab import userdata
import gradio as gr

#Retrieval Augmented Generation

###Carregamento de Arquivos

In [4]:
documents = SimpleDirectoryReader(input_dir='pdf')
print(documents.input_files)
documents = documents.load_data()
print(documents[0].__dict__)
print("-"*20)
print(documents[0].get_metadata_str())
print("-"*20)

[PosixPath('/content/pdf/serenatto_cafes_especiais.pdf')]
{'id_': '1a81895a-f64c-407f-b949-f7f65141857f', 'embedding': None, 'metadata': {'page_label': '1', 'file_name': 'serenatto_cafes_especiais.pdf', 'file_path': '/content/pdf/serenatto_cafes_especiais.pdf', 'file_type': 'application/pdf', 'file_size': 133957, 'creation_date': '2026-09-12', 'last_modified_date': '2026-09-12'}, 'excluded_embed_metadata_keys': ['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], 'excluded_llm_metadata_keys': ['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], 'relationships': {}, 'metadata_template': '{key}: {value}', 'metadata_separator': '\n', 'text_resource': MediaResource(embeddings=None, data=None, text='Serenatto\n \n–\n \nLoja\n \nonline\n \nde\n \ncafés\n \nespeciais\n \n \nInformações\n \ngerais\n \n \nP:\n \nQuais\n \nprodutos\n \nsão\n \nvendidos\n \nno\n \nsite\n \nda\n \nSerenatto?\n \nR:\n 

###Criando Chunks dos documentos

Chunking é o processo de dividir um texto extenso em partes menores e mais gerenciáveis, chamadas de chunks. Esses pedaços podem ser frases, parágrafos ou trechos de tamanho fixo, dependendo da aplicação.

In [5]:
# Foram feitos chunks de um valor específico, e é sempre bom avaliar
# se os chunks estão bem delimitados para não haver perda de contexto

# Quando haverá a segmentação
node_parser = SentenceSplitter(chunk_size=1200)

# Processo de chunk
nodes = node_parser.get_nodes_from_documents(documents, show_progress=True)
print(len(nodes))
print(nodes[0])

Parsing nodes:   0%|          | 0/10 [00:00<?, ?it/s]

11
Node ID: 5d980ec8-7314-446b-a6c2-a3260897f95b
Text: Serenatto   –   Loja   online   de   cafés   especiais
Informações   gerais     P:   Quais   produtos   são   vendidos   no
site   da   Serenatto?   R:   A   Serenatto   é   uma   loja   online
especializada   na   venda   de   grãos   de   cafés   especiais,
oferecendo   grãos   torrados,   provenientes   de   fazendas
selecionadas ...


###Armazenando em banco de dados local

*   Como o LlamaIndex usa OpenAI e há custos envolvidos, foi adaptado uma classe para aceitar o HuggingFace. Como HuggingFace e ChromaDB não tem total compatibilidade, a classe criada garante o uso dos conjuntos.
*   Cria-se o banco de dados dos vetores com o chromaDB. Esses embeddings serão então adicionados posteriormente.
*   Faz-se um embedded (conversão de texto em vetores), para armazenar os dados no banco juntamente com os índice dos vetores.



In [6]:
# Classe personalizada para adaptar a assinatura esperada pelo Chroma
class ChromaEmbeddingWrapper:
    def __init__(self, model_name): # Inicializa o modelo de embeddings do Hugging Face com o nome especificado
        self.model = HuggingFaceEmbedding(model_name=model_name)
        name = model_name

    def __call__(self, input): # Converte a entrada para um formato compatível com o HuggingFaceEmbedding
        return self.model.embed(input)

    def name(self):
        # propriedade 'name' adicionada ao Wrapper
        # para não gerar erro: "AttributeError: 'ChromaEmbeddingsWrapper' object has no attribute 'name'""
        return "ChromaE5Wrapper"

# Definindo o modelo de embedding usado pelo chroma
embed_model_chroma = ChromaEmbeddingWrapper("intfloat/multilingual-e5-large")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [7]:
# Criando um cliente persistente do ChromaDB, armazenando os dados no diretório './chroma_db'
db = chromadb.PersistentClient(path='./chroma_db')

# Atribuindo o cliente a uma variável para uso posterior
chroma_client = db

# Definindo o nome da coleção que será criada
collection_name = 'documentos'

# Tenta obter uma coleção existente ou criar uma nova, caso não exista
try:
  chroma_collection = chroma_client.get_or_create_collection(
      name = collection_name,
      embedding_function = embed_model_chroma
  )
# Captura e exibe qualquer erro que ocorra durante a criação ou carregamento da coleção
except Exception as e:
  print(f'Erro ao carregar ou criar coleção: {e}')

In [8]:
# Criando uma instância do ChromaVectorStore para armazenamento de vetores
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Configurando um contexto de armazenamento com o vector_store como armazenamento de vetores
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Definindo o modelo de embedding
embed_model = HuggingFaceEmbedding(model_name='intfloat/multilingual-e5-large')

# Criando um índice de armazenamento de vetores
index = VectorStoreIndex(nodes, storage_context=storage_context, embed_model=embed_model)

# Carregando um índice existente a partir do contexto de armazenamento
index = load_index_from_storage(storage_context, embed_model=embed_model)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# Criando o Chatbot com LLM Groq

In [9]:
# Chave Groq salva no Colab
GROQ_API = userdata.get('GROQ_API')

# Modelo de LLM - Groq OpenAI
llm = Groq(model="openai/gpt-oss-20b", api_key=GROQ_API)

# Responsável por processar consultas através dos índices
query_engine = index.as_query_engine(llm=llm, similarity_top_k=1)

In [10]:
# Fazendo pergunta ao chatbot e obtendo apenas a resposta.
response = query_engine.query('Quais grãos estão disponíveis?').response
print(response)
print("-" * 30)

# Vetores do texto da pergunta feita à llm
query_embedding = embed_model.get_text_embedding(response)
print(query_embedding)
print("-" * 30)

# Informações sobre a resposta associada aos vetores mais relevantes.
collection = chroma_collection.query(query_embedding, n_results=2, include=['distances', 'embeddings'])
print(collection)

Os grãos disponíveis são:

- Bourbon vermelho  
- Bourbon amarelo  
- Blend especial (mistura de Bourbon amarelo e vermelho)  
- Catuaí amarelo  
- Geisha  
- Yirgacheffe
------------------------------
[0.031836025416851044, -0.009940939024090767, 0.0004125343984924257, -0.03975595161318779, 0.017634063959121704, -0.04496762156486511, -0.0338825061917305, 0.08964689821004868, 0.0596606507897377, -0.03413381054997444, 0.0612279511988163, 0.021408043801784515, -0.05419502407312393, -0.001053712796419859, -0.019438238814473152, -0.013629014603793621, -0.016358671709895134, 0.03810667246580124, -0.002790669444948435, -0.0038910212460905313, 0.02387172356247902, 0.0016472621355205774, -0.033045023679733276, -0.022794466465711594, -0.03354046866297722, 0.0018776152282953262, -0.05107327923178673, -0.03433997929096222, -0.007905448786914349, 0.019420383498072624, -0.016360988840460777, 0.027912436053156853, -0.04445713013410568, -0.015704289078712463, -0.01948464848101139, 0.00117810152005404

# Refinando o chat com contexto ampliado

1.   Entrada inicial do usuário
2.   Consulta / busca por informações relevantes
3.    Retorno das informações
4.    Entrada inicial + contexto ampliado
5.    Resposta gerada pelo modelo

## Query Engine x Chat Engine

### Query Engine: consultas diretas e específicas
O Query Engine é uma ferramenta projetada para realizar consultas diretas e pontuais em um índice de dados. Ele é ideal para recuperar informações específicas de forma rápida e eficiente, como detalhes sobre produtos, preços ou políticas da empresa. No entanto, ele não mantém o contexto de uma conversa, sendo mais adequado para perguntas isoladas e objetivas.

### Chat Engine: interações dinâmicas e contextualizadas
Já o Chat Engine foi desenvolvido para criar interações mais dinâmicas e contextualizadas, semelhantes a uma conversa humana. Ele é a escolha ideal para o chatbot da Serenatto, pois permite manter o contexto da conversa ao longo do tempo, responder a perguntas de acompanhamento e oferecer uma experiência de diálogo fluida e natural. Com o Chat Engine, o chatbot pode, por exemplo, ajudar um cliente a escolher um café com base em suas preferências, sugerir combinações ou esclarecer dúvidas sobre métodos de preparo.

In [11]:
# Gerenciamento do histórico do chat. Será aplicado para limitar o número
# de tokens consumidos, garantindo contexto sem sobrecarregar a memória.
# Limitando o histórico a 256 tokens
memory = ChatSummaryMemoryBuffer(llm=llm, token_limit=256)

# O as_chat_engine no modo 'context' recupera textos relevantes do
# índice (index) com base na mensagem do usuário, define esses textos
# como contexto no sistema e gera uma resposta contextualizada.
chat_engine = index.as_chat_engine(
    chat_mode='context',
    llm=llm,
    memory=memory,
    system_prompt=('''Você é especialista em cafés especiais da Serenatto,
    uma loja online que vende grãos de cafés torrados. Sua função é tirar
    dúvidas de forma simpática e natural sobre os grãos disponíveis''')
)

pergunta = chat_engine.chat('Você pode me dar mais detalhes sobre o catuai amarelo?').response
print(pergunta)
print("-"*30)
log_chat = chat_engine.chat_history
print(log_chat)
print("-"*30)
print(memory.get())

# resetando a memória do chat
# chat_engine.reset()

Claro! O **Catuaí Amarelo** é um dos nossos cafés mais vibrantes e, se você gosta de algo que se destaque, ele é uma escolha fantástica. Aqui vai um panorama completo:

| **Aspecto** | **Detalhes** |
|-------------|--------------|
| **Origem** | Grãos cultivados em altitudes de 1.200 m, principalmente em regiões de clima tropical úmido (Minas Gerais, Bahia e Espírito Santo). A altitude confere ao café uma maturação mais lenta, o que intensifica a complexidade dos sabores. |
| **Processo** | Fermentação natural (wet‑hopping) seguida de secagem ao sol. Esse método realça a acidez viva e permite que os sabores frutados se desenvolvam de forma mais pronunciada. |
| **Torragem** | Média (aprox. 12–14 minutos a 210 °C). Mantém a acidez brilhante e abre as notas de fruta, sem perder a suavidade do corpo. |
| **Perfil Sensorial** | • **Doçura**: 5/5 – sensação de doçura suave, quase caramelizada. <br>• **Acidez**: 4–5/5 – viva, com notas de frutas cítricas e tropical. <br>• **Corpo**: 4/5 – mé

# Criando interface com Gradio

In [12]:
# Criando a função para conversar com o chatbot
def converse_com_bot(message, chat_history):
    response = chat_engine.chat(message)

    if chat_history is None:
        chat_history = []

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": response.response})

    return "", chat_history

# Criando a função para resetar o chatbot
def resetar_chat():
  chat_engine.reset()
  return []

In [13]:
with gr.Blocks() as demo:
  gr.Markdown('# Chatbot da Serenatto')

  chatbot = gr.Chatbot()
  msg = gr.Textbox(label='Digite a sua mensagem')
  limpar = gr.Button('Limpar')

  msg.submit(converse_com_bot, [msg, chatbot], [msg, chatbot])
  limpar.click(resetar_chat, None, chatbot, queue=False)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cb37c9531f83ad8c1a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cb37c9531f83ad8c1a.gradio.live
